In [1]:
import pandas as pd

In [2]:
link_data='https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet'

In [3]:
columns = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 
           'trip_distance', 'tip_amount', 'total_amount']
df = pd.read_parquet(link_data, columns=columns)#.head(100)
df.head(5)

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [4]:
df['passenger_count'] = df['passenger_count'].fillna(0).astype(int)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49416 entries, 0 to 49415
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   lpep_pickup_datetime   49416 non-null  datetime64[us]
 1   lpep_dropoff_datetime  49416 non-null  datetime64[us]
 2   PULocationID           49416 non-null  int32         
 3   DOLocationID           49416 non-null  int32         
 4   passenger_count        49416 non-null  int64         
 5   trip_distance          49416 non-null  float64       
 6   tip_amount             49416 non-null  float64       
 7   total_amount           49416 non-null  float64       
dtypes: datetime64[us](2), float64(3), int32(2), int64(1)
memory usage: 2.6 MB


In [6]:
from dataclasses import dataclass

@dataclass
class Ride:
    lpep_pickup_datetime: int  # epoch milliseconds
    lpep_dropoff_datetime: int  # epoch milliseconds
    PULocationID: int
    DOLocationID: int
    passenger_count: int
    trip_distance: float
    tip_amount: float
    total_amount: float

In [7]:
def ride_from_row(row):
    return Ride(
        lpep_pickup_datetime=int(row['lpep_pickup_datetime'].timestamp() * 1000),
        lpep_dropoff_datetime=int(row['lpep_dropoff_datetime'].timestamp() * 1000),
        PULocationID=int(row['PULocationID']),
        DOLocationID=int(row['DOLocationID']),
        passenger_count=int(row['passenger_count']),
        trip_distance=float(row['trip_distance']),
        tip_amount=float(row['tip_amount']),
        total_amount=float(row['total_amount']),
    )

In [ ]:
# ride = ride_from_row(df.iloc[0])
# ride

#### Convert data

In [8]:
import json

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

In [9]:
from kafka import KafkaProducer

server = 'redpanda:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

In [10]:
import dataclasses

# topic_name = 'rides'
topic_name = 'green-trips'

# producer.send(topic_name, value=dataclasses.asdict(ride))
# producer.flush()

In [11]:
def ride_serializer(ride):
    ride_dict = dataclasses.asdict(ride)
    json_str = json.dumps(ride_dict)
    return json_str.encode('utf-8')

In [12]:
producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

#### Load test data - 1 row

In [ ]:
# producer.send(topic_name, value=ride)
# producer.flush()

#### Produce all data

In [13]:
import time

t0 = time.time()

# for _, row in df.iterrows():
for i, (_, row) in enumerate(df.iterrows()):
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    # print(f"Sent: {ride}")
    if i % 5000 == 0:
        print(f"Progress: {i}/{len(df)} rows sent...")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

Progress: 0/49416 rows sent...
Progress: 5000/49416 rows sent...
Progress: 10000/49416 rows sent...
Progress: 15000/49416 rows sent...
Progress: 20000/49416 rows sent...
Progress: 25000/49416 rows sent...
Progress: 30000/49416 rows sent...
Progress: 35000/49416 rows sent...
Progress: 40000/49416 rows sent...
Progress: 45000/49416 rows sent...
took 538.89 seconds
